<a href="https://colab.research.google.com/github/zsgwu/G5_GWU_CAPST/blob/main/get_embeddings_eduyou.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EduYou — Get Embeddings (Azure OpenAI)

**File:** `get_embeddings_eduyou.ipynb`  
**Generated:** 2026-03-30

This notebook is a drop-in adaptation of the professor’s *get_embeddings* template for the EduYou capstone. It creates vector embeddings for **one row per document** from:

- `cleaned/eduyou_cip_docs_for_embedding.csv`

and writes an embeddings table compatible with the professor’s `07_RAG_query.ipynb` pattern (doc_id + `dim_*` columns).

---

## Column-by-column mapping (EduYou → RAG template)

| EduYou CSV column | Role in RAG template | Used how |
|---|---|---|
| `text` | document text | embedded into a vector |
| `doc_id` | document identifier | stored alongside embedding |
| `cip4`, `degree_level`, `cip_title`, `median_earnings_4yr_nat` | metadata | carried into output for filtering/debugging |

> ✅ **Important:** Do *not* embed the raw multi-million-row join tables. Use the document-level file (`eduyou_cip_docs_for_embedding.csv`).

---

## Embedding model configuration (Professor note)

Set `AZURE_OPENAI_DEPLOYMENT_ID` to one of:
- `text-embedding-ada-002`
- `text-embedding-3-small`
- `text-embedding-3-large`

Deployments are created using the same names as the models, so you can use those names directly.


In [14]:
from google.colab import drive
drive.mount('/content/drive')

import os
print('Working dir:', os.getcwd())
print('Files:', os.listdir())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working dir: /content
Files: ['.config', 'embeddings', 'drive', 'sample_data']


## 1) Install dependencies


In [13]:
!pip -q install openai pandas numpy

## 2) Configuration

Set your Azure OpenAI endpoint and key securely.
- Recommended: store in environment variables.
- Do **not** hardcode secrets in notebooks.


In [15]:
import os
import getpass
import pandas as pd
import numpy as np
from openai import AzureOpenAI

# --- Required settings ---
AZURE_OPENAI_ENDPOINT = (
    os.getenv('AZURE_OPENAI_ENDPOINT')
    or getpass.getpass('AZURE_OPENAI_ENDPOINT (e.g., https://<resource>.openai.azure.com/): ')
)
AZURE_OPENAI_API_KEY = (
    os.getenv('AZURE_OPENAI_API_KEY')
    or getpass.getpass('AZURE_OPENAI_API_KEY: ')
)

# ✅ Updated to match professor instructions
AZURE_OPENAI_API_VERSION = (
    os.getenv('AZURE_OPENAI_API_VERSION')
    or '2025-04-01-preview'
)

# --- Deployment/model name (per professor note) ---
AZURE_OPENAI_DEPLOYMENT_ID = (
    os.getenv('AZURE_OPENAI_DEPLOYMENT_ID')
    or 'text-embedding-3-small'
)

# Optional: for text-embedding-3-* you may set dimensions to shorten vectors.
# Leave as None to use model default (3-small: 1536, 3-large: 3072).
EMBEDDING_DIMENSIONS = None

# Input/output paths
INPUT_DOCS_CSV = '/content/drive/MyDrive/group-5/RAG_data/cleaned/eduyou_cip_docs_for_embedding.csv'
OUTPUT_DIR = 'embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_EMB_CSV = os.path.join(
    OUTPUT_DIR, f"eduyou_embeddings_{AZURE_OPENAI_DEPLOYMENT_ID}.csv"
)

print('Model/deployment:', AZURE_OPENAI_DEPLOYMENT_ID)
print('Input docs:', INPUT_DOCS_CSV)
print('Output embeddings:', OUTPUT_EMB_CSV)



AZURE_OPENAI_ENDPOINT (e.g., https://<resource>.openai.azure.com/): ··········
AZURE_OPENAI_API_KEY: ··········
Model/deployment: text-embedding-3-small
Input docs: /content/drive/MyDrive/group-5/RAG_data/cleaned/eduyou_cip_docs_for_embedding.csv
Output embeddings: embeddings/eduyou_embeddings_text-embedding-3-small.csv


## 3) Load EduYou document table

This file must have at minimum:
- `doc_id`
- `text`


In [16]:
docs = pd.read_csv(INPUT_DOCS_CSV, low_memory=False)

required_cols = {'doc_id', 'text'}
missing = required_cols - set(docs.columns)
if missing:
    raise ValueError(f"Missing required columns in {INPUT_DOCS_CSV}: {missing}")

print('Rows (documents):', len(docs))
print('Columns:', docs.columns.tolist())

Rows (documents): 1312
Columns: ['doc_id', 'cip4', 'degree_level', 'cip_title', 'median_earnings_4yr_nat', 'text']


## 4) Create embeddings (batched)

This follows the professor’s Azure OpenAI embedding example using `AzureOpenAI`.
We batch requests to reduce overhead.

The output file uses:
- `doc_id`
- metadata columns (if present)
- embedding columns named `dim_0 ... dim_{p-1}`

The notebook can resume if partially completed.


In [17]:
# client initialization
client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT
)

# Metadata columns to carry through if present
meta_cols = [
    c for c in ['cip4', 'degree_level', 'cip_title', 'median_earnings_4yr_nat']
    if c in docs.columns
]

BATCH_SIZE = 64

# Resume support
if os.path.exists(OUTPUT_EMB_CSV):
    emb_df = pd.read_csv(OUTPUT_EMB_CSV)
    done_ids = set(emb_df['doc_id'].astype(str))
    print(f"Found existing embeddings file with {len(done_ids)} rows. Skipping completed docs.")
else:
    done_ids = set()

# Helper to call embeddings API
def get_embeddings_batch(text_list):
    kwargs = {
        'model': AZURE_OPENAI_DEPLOYMENT_ID,
        'input': text_list,
    }
    if EMBEDDING_DIMENSIONS is not None:
        kwargs['dimensions'] = int(EMBEDDING_DIMENSIONS)

    resp = client.embeddings.create(**kwargs)
    return [d.embedding for d in resp.data]

# Determine embedding dimension once
first_vec = get_embeddings_batch([docs.loc[0, 'text']])[0]
P = len(first_vec)
print('Embedding dimension P =', P)

dim_cols = [f'dim_{i}' for i in range(P)]

# Initialize output file if needed
if not os.path.exists(OUTPUT_EMB_CSV):
    cols = ['doc_id'] + meta_cols + dim_cols
    pd.DataFrame(columns=cols).to_csv(OUTPUT_EMB_CSV, index=False)

# ✅ FIXED batch loop (no overlapping batches)
for batch_start in range(0, len(docs), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(docs))
    batch_rows = docs.iloc[batch_start:batch_end].copy()

    batch_rows['doc_id'] = batch_rows['doc_id'].astype(str)
    batch_rows = batch_rows[~batch_rows['doc_id'].isin(done_ids)]

    if batch_rows.empty:
        continue

    texts = batch_rows['text'].astype(str).tolist()
    vecs = get_embeddings_batch(texts)

    out_rows = []
    for (_, row), vec in zip(batch_rows.iterrows(), vecs):
        rec = {'doc_id': row['doc_id']}
        for mc in meta_cols:
            rec[mc] = row.get(mc, '')
        rec.update({dim_cols[j]: float(vec[j]) for j in range(P)})
        out_rows.append(rec)
        done_ids.add(row['doc_id'])

    pd.DataFrame(out_rows).to_csv(
        OUTPUT_EMB_CSV, mode='a', header=False, index=False
    )

print('✅ Done. Saved embeddings to:', OUTPUT_EMB_CSV)

Found existing embeddings file with 1312 rows. Skipping completed docs.
Embedding dimension P = 1536
✅ Done. Saved embeddings to: embeddings/eduyou_embeddings_text-embedding-3-small.csv


## 5) Quick sanity check

Load the embeddings output and confirm shape.


In [18]:
emb = pd.read_csv(OUTPUT_EMB_CSV)
print('Embedding table shape:', emb.shape)
emb.head()


Embedding table shape: (1312, 1541)


,doc_id,cip4,degree_level,cip_title,median_earnings_4yr_nat,dim_0,dim_1,dim_2,dim_3,dim_4,...,dim_1526,dim_1527,dim_1528,dim_1529,dim_1530,dim_1531,dim_1532,dim_1533,dim_1534,dim_1535
0,1001_Associates_Degree,1001,Associate's Degree,Communications Technologies/Technicians.,35292.0,-0.011399,0.001650,0.032297,0.000320,-0.044620,...,0.018434,0.030218,-0.017548,-0.014339,0.033992,0.001988,-0.027214,-0.042618,-0.024236,0.036585
1,1001_Bachelors_Degree,1001,Bachelor's Degree,Communications Technologies/Technicians.,36451.0,-0.006818,0.004062,0.037305,0.001938,-0.054646,...,0.019463,0.027683,-0.021444,-0.010606,0.035016,0.002843,-0.023515,-0.046696,-0.020312,0.036714
2,1001_Masters_Degree,1001,Master's Degree,Communications Technologies/Technicians.,71506.0,-0.006827,0.002901,0.032675,-0.007824,-0.056459,...,0.018917,0.034667,-0.014613,-0.009874,0.035587,-0.001236,-0.025228,-0.047338,-0.017436,0.031448
3,1001_Undergraduate_Certificate_or_Diploma,1001,Undergraduate Certificate or Diploma,Communications Technologies/Technicians.,33445.0,-0.004809,0.004246,0.039976,-0.000188,-0.054630,...,0.020027,0.029076,-0.022901,-0.011489,0.030551,0.001735,-0.021878,-0.043186,-0.020027,0.037464
4,1002_Associates_Degree,1002,Associate's Degree,Audiovisual Communications Technologies/Techni...,35660.0,-0.032617,-0.000759,0.027159,-0.003912,-0.054214,...,0.019942,0.031194,-0.024870,-0.011310,0.044670,-0.005076,-0.031504,-0.045705,-0.019050,0.033005


## Next step

Use the resulting embeddings CSV in the professor’s `07_RAG_query.ipynb` workflow:
- Load this embeddings file
- Load the same document file (`eduyou_cip_docs_for_embedding.csv`)
- Compute query embedding with the same `AZURE_OPENAI_DEPLOYMENT_ID`
- Retrieve Top‑K by distance/similarity
